In [ ]:
!pip -q install faiss-cpu sentence-transformers transformers accelerate bitsandbytes datasets tiktoken unstructured[html] readability-lxml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 45.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.6/167.6 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.8/207.8 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.6/114.6 kB 11.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/college_chatbot"
import os; os.makedirs(SAVE_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
from huggingface_hub import login

# Paste your HF token here (you get it from https://huggingface.co/settings/tokens)
login("")


In [ ]:
import os, re, time, json, math, urllib.parse, requests
from collections import deque
from bs4 import BeautifulSoup
from readability import Document
import numpy as np
import faiss
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, CrossEncoder
import tiktoken  # only used for conservative token counting; falls back to words if needed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

############################
# Config
############################
SEED_URL      = "https://manipal.edu/mu.html"   # change this to your college root page
MAX_PAGES     = 120                              # hard cap
MAX_DEPTH     = 3                                # link depth
SAME_DOMAIN   = True                             # stay inside base domain
CHUNK_TOKENS  = 180                              # per chunk (RAG-friendly)
CHUNK_OVERLAP = 30                               # soft overlap to keep context continuity
TOP_N         = 12                               # fetch N, then rerank
TOP_K         = 4                                # feed K to LLM
MODEL_NAME    = "mistralai/Mistral-7B-Instruct-v0.2"  # long context
CTX_TOKENS    = 8192                             # model context window
ANSWER_LEN    = 220                              # ~3-4 sentences
EMBED_MODEL   = "BAAI/bge-small-en-v1.5"         # great quality/speed trade-off
RERANK_MODEL  = "cross-encoder/ms-marco-MiniLM-L-6-v2" # light reranker

############################
# Helpers
############################
def same_domain(url, root):
    return urllib.parse.urlparse(url).netloc == urllib.parse.urlparse(root).netloc

def clean_url(href, base):
    return urllib.parse.urljoin(base, href.split("#")[0])

def fetch(url, timeout=15):
    try:
        r = requests.get(url, timeout=timeout, headers={"User-Agent":"Mozilla/5.0"})
        if "text/html" not in r.headers.get("content-type",""): return None
        return r.text
    except Exception:
        return None

def extract_text(html):
    if not html: return ""
    # Use readability to get main article content; fallback to soup text
    try:
        doc = Document(html)
        main_html = doc.summary()
        soup = BeautifulSoup(main_html, "lxml")
    except Exception:
        soup = BeautifulSoup(html, "lxml")
    # keep paragraphs & headings
    for tag in soup(["script","style","noscript","header","footer","nav","aside","form"]):
        tag.decompose()
    text = []
    for el in soup.find_all(["h1","h2","h3","h4","p","li"]):
        t = re.sub(r"\s+", " ", el.get_text(" ", strip=True))
        if len(t) >= 5:
            text.append(t)
    return "\n".join(text)

def tokenize_len(text, enc_name="cl100k_base"):
    try:
        enc = tiktoken.get_encoding(enc_name)
        return len(enc.encode(text))
    except Exception:
        return len(text.split())

def chunk_by_tokens(text, chunk_tokens=CHUNK_TOKENS, overlap=CHUNK_OVERLAP):
    # simple token-aware chunking
    words = text.split()
    approx_tok_per_word = 0.75  # rough heuristic for English
    step = max(1, int((chunk_tokens - overlap) / approx_tok_per_word))
    size = max(1, int(chunk_tokens / approx_tok_per_word))
    chunks = []
    for i in range(0, len(words), step):
        piece = " ".join(words[i:i+size])
        if piece.strip():
            chunks.append(piece)
    return chunks

############################
# Crawl
############################
def crawl(seed_url, max_pages=MAX_PAGES, max_depth=MAX_DEPTH, same_dom=SAME_DOMAIN):
    seen, out = set(), []
    dq = deque([(seed_url, 0)])
    base_netloc = urllib.parse.urlparse(seed_url).netloc
    while dq and len(out) < max_pages:
        url, d = dq.popleft()
        if url in seen or (same_dom and urllib.parse.urlparse(url).netloc != base_netloc):
            continue
        seen.add(url)
        html = fetch(url)
        if not html: continue
        text = extract_text(html)
        if len(text.split()) > 50:  # keep only non-trivial pages
            out.append({"url": url, "text": text})
        if d < max_depth:
            soup = BeautifulSoup(html, "lxml")
            for a in soup.find_all("a", href=True):
                new = clean_url(a["href"], url)
                if new.startswith("mailto:") or new.startswith("tel:"): continue
                dq.append((new, d+1))
    return out

print("🕸️  Crawling site… this may take a few minutes.")
docs = crawl(SEED_URL)

############################
# Build index (persist if desired)
############################
print(f"📄 Collected pages: {len(docs)}")
raw_chunks, meta = [], []
for d in docs:
    chunks = chunk_by_tokens(d["text"])
    for c in chunks:
        raw_chunks.append(c)
        meta.append({"url": d["url"]})

print(f"🔪 Created chunks: {len(raw_chunks)}")

embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
emb = embedder.encode(raw_chunks, batch_size=64, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb.astype("float32"))

# Save index & metadata
if 'SAVE_DIR' in globals():
    faiss.write_index(index, os.path.join(SAVE_DIR, "faiss.index"))
    with open(os.path.join(SAVE_DIR, "chunks.json"), "w") as f:
        json.dump(raw_chunks, f)
    with open(os.path.join(SAVE_DIR, "meta.json"), "w") as f:
        json.dump(meta, f)
    print("💾 Index saved to Drive.")

############################
# Reranker (optional)
############################
reranker = CrossEncoder(RERANK_MODEL, device=DEVICE)

def retrieve(query, top_n=TOP_N, top_k=TOP_K):
    q = embedder.encode([query], normalize_embeddings=True)
    D, I = index.search(q, top_n)
    cands = [(raw_chunks[i], meta[i]) for i in I[0] if 0 <= i < len(raw_chunks)]
    # rerank by cross-encoder
    pairs = [[query, c[0]] for c in cands]
    scores = reranker.predict(pairs)
    reranked = [c for _, c in sorted(zip(scores, cands), key=lambda x: x[0], reverse=True)]
    return reranked[:top_k]

############################
# Load LLM (4-bit, long context)
############################
print(f"🧠 Loading {MODEL_NAME} in 4-bit on {DEVICE}…")
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.model_max_length = CTX_TOKENS

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_cfg,
    device_map="auto"
)

############################
# Prompting & generation
############################
SYSTEM = (
    "You are a helpful college information assistant. "
    "Answer using ONLY the provided context. If the answer is not in the context, say: \"I don't know.\" "
    "Keep answers concise (about 3–4 sentences)."
)

def fit_context(query, blocks, max_ctx_tokens=CTX_TOKENS, safety_margin=768):
    # assemble context until we are within limits
    pieces, total = [], 0
    for text, m in blocks:
        t = f"[Source: {m['url']}]\n{text}\n"
        tlen = tokenize_len(t)
        if total + tlen > (max_ctx_tokens - safety_margin):
            break
        pieces.append(t)
        total += tlen
    return "\n".join(pieces)

def build_prompt(query, blocks):
    context = fit_context(query, blocks)
    return f"{SYSTEM}\n\nContext:\n{context}\n\nUser question: {query}\nAnswer:"

@torch.inference_mode()
def answer(query):
    blocks = retrieve(query)
    prompt = build_prompt(query, blocks)
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=CTX_TOKENS).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=ANSWER_LEN,
        do_sample=True,
        temperature=0.6,
        top_p=0.9,
        repetition_penalty=1.1,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.eos_token_id,
    )
    text = tok.decode(out[0], skip_special_tokens=True)
    # strip prompt
    prompt_text = tok.decode(inputs["input_ids"][0], skip_special_tokens=True)
    return text[len(prompt_text):].strip()

print("✅ RAG chatbot ready.")

🕸️  Crawling site… this may take a few minutes.
📄 Collected pages: 120
🔪 Created chunks: 231


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

✅ RAG chatbot ready.


In [ ]:
# Try: "Who is the Vice Chancellor?", "List key departments", "What is the admission process?"
while True:
    q = input("You: ").strip()
    if not q:
        print("Bot: Goodbye!")
        break
    print("Bot:", answer(q), "\n")

You: Who is the vice Chancellor of Manipal?
Bot: The Vice-Chancellor of Manipal Academy of Higher Education is currently not mentioned in the provided context. The Pro Chancellor is identified as the Vice-Chancellor in the context of the University Sports Council, but no name is given. Therefore, I don't know who the Vice-Chancellor of Manipal is based on the provided context. 

You: who is the vice Chancellor of MAHE
Bot: Lt. Gen. (Dr.) M. D. Venkatesh, VSM (Retd) is the Vice Chancellor of MAHE. 

You: who is the director of MIT BLR?
Bot: The Director of MIT Bengaluru is not mentioned in the context. However, there is a Director PR, Media & Social Media Cell for MAHE Bengaluru. 

You: what can you answer about MAHE BLR?
Bot: MAHE Bangalore Campus offers programmes in Regenerative Medicine, Engineering, Art & Design, Humanities, Liberal Arts, Social Sciences, Law, and Management. It is one of the campuses of Manipal Academy of Higher Education (MAHE), which is an Institution of Eminenc